# 06b — Canary causale dello state updater ottimizzato

Verifica multi-seed dell'updater atomico a 1 ms. Il braccio primario usa soltanto `V_t` e input causali; il braccio con endpoint teacher è esclusivamente un riferimento diagnostico. Non vengono letti validation, test o microtracce future.

In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');ROOT=Path('/kaggle/working');ELM_REPO=ROOT/'hayflow_workspace'/'elmneuron';ELM_REPO.parent.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO);REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches();print({'revision':REVISION})

## 1. Autorità e dataset

Caricare come Input Kaggle il risultato 06a-b, l'autorità 05t, il dataset targeted base e il top-up BAP v3. La selezione degli artefatti usa l'hash dell'indice, non il nome assegnato da Kaggle.

In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source
from src.hayflow_model.atomic_state_dynamics_playground import EXPECTED_05T_INDEX_SHA256
from src.hayflow_model.optimized_explicit_state_updater_canary import EXPECTED_06AB_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input')
source05t=os.environ.get('HAYFLOW_05T_ARTIFACT');ARTIFACT_05T_SOURCE=discover_indexed_artifact_source(INPUT_ROOT,EXPECTED_05T_INDEX_SHA256,override=Path(source05t) if source05t else None);assert ARTIFACT_05T_SOURCE is not None,'Artefatto 05t esatto non trovato.'
source06ab=os.environ.get('HAYFLOW_06AB_ARTIFACT');ARTIFACT_06AB_SOURCE=discover_indexed_artifact_source(INPUT_ROOT,EXPECTED_06AB_INDEX_SHA256,override=Path(source06ab) if source06ab else None);assert ARTIFACT_06AB_SOURCE is not None,'Artefatto 06a-b esatto non trovato: aggiungi hayflow_atomic_voltage_path_identifiability agli Input Kaggle.'
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_stamp';stamp=f'{source.stat().st_size}:{source.stat().st_mtime_ns}'
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow06b_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.';print({'05t':str(ARTIFACT_05T_SOURCE),'06a_b':str(ARTIFACT_06AB_SOURCE),'base':str(BASE_SOURCE),'composite_manifest':str(COMPOSITE_MANIFEST)})

In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 06b][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880 and not bundle.manifest['physical_merge_performed'];print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})

## 2. Preflight causale

Ricostruisce i ruoli train-only e verifica che il braccio primario non legga endpoint o microtracce. L'output mostra soltanto il contratto essenziale.

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import OptimizedExplicitStateCanaryConfig,OptimizedExplicitStateUpdaterCanary
cfg=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_optimized_explicit_state_updater_canary.yml').read_text());config=OptimizedExplicitStateCanaryConfig.from_mapping(cfg['optimized_explicit_state_updater_canary'])
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_optimized_explicit_state_updater_canary');assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
session=OptimizedExplicitStateUpdaterCanary(bundle,OUTPUT_DIR,config,ARTIFACT_05T_SOURCE,ARTIFACT_06AB_SOURCE,code_revision=REVISION);preflight=session.prepare_causal_canary();display({'valid':preflight['valid'],'roles':preflight['role_transition_counts'],'arms':preflight['arms'],'seeds':preflight['pilot_seeds'],'steps_per_run':preflight['training_steps_per_seed_and_arm'],'parameter_count':preflight['parameter_count'],'parameter_ceiling':preflight['parameter_ceiling'],'future_microtraces_read':preflight['future_microtraces_read'],'primary_uses_teacher_endpoint':preflight['teacher_endpoint_read_by_primary_arm'],'state_splits_read':preflight['state_and_outcome_splits_read']});assert preflight['valid'] and preflight['state_and_outcome_splits_read']==['train'] and not preflight['future_microtraces_read'] and not preflight['teacher_endpoint_read_by_primary_arm'] and not preflight['validation_state_accessed'] and not preflight['test_state_accessed']

## 3. Sei run appaiati e rollout annidati

Tre seed per due bracci, 1.200 step ciascuno. Il tracker stampa una riga ogni 200 step e non visualizza tensori, RNG o dizionari completi.

In [ ]:
try:
 canary_report=session.run_causal_canary();rollout_report=session.evaluate_causal_nested_rollouts();final_report=session.finalize_causal_canary(canary_report,rollout_report)
finally:
 session.close()
seed_summary={seed:{'causal_gain':round(row['causal_start_voltage']['one_step_gain'],4),'endpoint_gain':round(row['linear_endpoint_path']['one_step_gain'],4),'causal_macro_gain':round(row['causal_start_voltage']['semantic_macro_gain'],4),'causal_active_gain':round(row['causal_start_voltage']['active_gain'],4),'positive_groups':round(row['causal_start_voltage']['positive_semantic_group_fraction'],3),'retention':round(row['causal_retention_vs_endpoint'],3),'causal_8ms':round(row['causal_start_voltage']['eight_ms_rollout_gain'],4)} for seed,row in final_report['per_seed'].items()}
aggregate={key:(round(value,4) if isinstance(value,(int,float)) else {k:round(v,4) for k,v in value.items()}) for key,value in final_report['aggregate'].items()}
display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'causal_confirmed':final_report['causal_updater_confirmed'],'per_seed':seed_summary,'aggregate':aggregate,'all_seed_gates':final_report['all_causal_per_seed_gates_passed'],'all_rollouts_positive':final_report['all_causal_rollout_gains_positive'],'next_step':final_report['next_step']});assert final_report['valid'] and final_report['rollout_windows_nested'] and final_report['same_parameter_count_across_all_runs'] and not final_report['future_microtraces_read'] and not final_report['primary_arm_uses_teacher_endpoint'] and not final_report['validation_state_accessed'] and not final_report['test_state_accessed']

In [ ]:
from IPython.display import Image,display
display(Image(filename=str(OUTPUT_DIR/'figures/optimized_causal_state_canary.png')))

## 4. Crea e scarica lo ZIP

Downloader browser stabile del progetto: ZIP in `/kaggle/working`, base64, `Blob` e click temporaneo.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_optimized_explicit_state_updater_canary','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})